# 1. Generate the Protocell-0 corpus

This notebook writes the text that every later notebook and lab learns from: the genomes of **Protocell-0**,
an invented organism whose genetic language we designed, so the ground truth is known exactly.

Each genome carries **one Nax operon**: the seven genes of the energy pathway (NaxGATE → NaxDUMP), each with
its ribosome-binding site, initiator, marker, body and terminator, plus a promoter, a kelate-sensing leader
before NaxSNIP and a transcription terminator. From each genome the cell **transcribes** RNA under one of two
conditions:

- `K_low` (kelate scarce): read-through, all seven genes;
- `K_high` (kelate abundant): transcription stops at the leader's attenuator, so only NaxGATE and NaxTAG.

The notebook produces:

| Output | Content |
|---|---|
| `data/corpus/genome.csv` | every codon of every genome, with labels |
| `data/corpus/genome-shuffled.csv` | the same genomes with the genes in shuffled order (a control) |
| `data/corpus/transcripts.csv` | every codon of every transcript, with labels |
| `data/corpus/proteins.csv` | the translated proteins |
| `data/web/language.json` | the language tables, for the web labs |
| `data/web/genome-sample.json` | 60 genomes, their transcripts and shuffled versions, for the labs |
| `data/web/showcase-transcripts.json` | six transcripts chosen to show the interesting cases |

The language itself (bases, code table, residues, reserved codons, genes and dialects) is **not** defined
here: it lives in `protocell_lang.py`, so no two notebooks can disagree about it. The specification is
`docs/design-spec.md` §1–§2 and §5.

*Run this notebook from its own directory (`01-corpus/`); the paths below are relative to it.*

## Configuration

All randomness comes from one seeded generator, so the corpus is reproducible: the same seed gives the same
genomes, codon for codon. The default values are those of the specification (§2, §4.2).

In [1]:
import sys, os, json, inspect, html
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from IPython.display import HTML, display

sys.path.insert(0, "..")          # protocell_lang.py lives in the project root
import protocell_lang as L

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
SEED = 42
N_GENOMES = 400
TRANSCRIPTS_PER_GENOME = 4

P_SPACER_DECOY = 1.0              # Poisson mean: bare WXZ decoys per genome
P_BROKEN_RBS = 0.15               # probability that a gene block has a mutated RBS
P_DECOY_MARKER = 0.5              # probability that a decoy WXZ is followed by a gene marker

FLANK_LENGTH = (10, 40)           # spacer before the promoter and after the transcription terminator
UTR5_LENGTH = (0, 2)
GAP_LENGTH = (1, 3)               # between the RBS and the initiator
BODY_LENGTH = (12, 24)
INTERGENIC_LENGTH = (0, 2)
LEADER_SPACER_LENGTH = (0, 2)

N_SAMPLE_GENOMES = 60             # genomes exported for the web labs

CORPUS_DIR = "../data/corpus"
WEB_DIR = "../data/web"
NOTEBOOK = "01-corpus/01_generate_corpus.ipynb"

PARAMS = {
    "n_genomes": N_GENOMES, "transcripts_per_genome": TRANSCRIPTS_PER_GENOME,
    "p_spacer_decoy": P_SPACER_DECOY, "p_broken_rbs": P_BROKEN_RBS, "p_decoy_marker": P_DECOY_MARKER,
    "flank_length": FLANK_LENGTH, "utr5_length": UTR5_LENGTH, "gap_length": GAP_LENGTH,
    "body_length": BODY_LENGTH, "intergenic_length": INTERGENIC_LENGTH,
    "leader_spacer_length": LEADER_SPACER_LENGTH, "operon_order": ["pathway", "shuffled"],
}

rng = np.random.default_rng(SEED)
os.makedirs(CORPUS_DIR, exist_ok=True)
os.makedirs(WEB_DIR, exist_ok=True)
print("protocell_lang version", L.GENERATOR_VERSION)

protocell_lang version 1.0.0


## 2. The language

Everything in this section is printed from `protocell_lang.py`.

**Bases.** Four DNA bases; in RNA, Z (zoltine) becomes V (vethine). The real analogues show that the
Protocell-0 code is a renamed copy of a real one.

In [2]:
pd.DataFrame(L.BASES).T

,name,pairs_with,bond,real_analogue,rna_only
W,welline,Z,weak,A,NaN
X,xoline,Y,strong,G,NaN
Y,ythrine,X,strong,C,NaN
Z,zoltine,W,weak,T,NaN
V,vethine,W,weak,U,True


**The code table (Stages I–II).** Codons of the form `[XY] N [XY]` code for residues: 16 codons, 10 residues.
Rows are the first·third base, columns the middle base. The middle base decides the chemistry; synonyms differ
only in the third base.

In [3]:
grid = pd.DataFrame(
    [[f"{f}{m}{t} → {L.CODE_TABLE[f + m + t]}" for m in L.CODE_GRID_COLUMNS] for f, t in L.CODE_GRID_ROWS],
    index=[f"{f}·{t}" + (" (Stage I)" if (f, t) == ("X", "Y") else "") for f, t in L.CODE_GRID_ROWS],
    columns=[f"middle {m}" for m in L.CODE_GRID_COLUMNS],
)
grid

,middle W,middle X,middle Y,middle Z
X·Y (Stage I),XWY → Ds,XXY → Nm,XYY → Kv,XZY → Th
X·X,XWX → My,XXX → Nm,XYX → Kv,XZX → Th
Y·Y,YWY → Os,YXY → Si,YYY → Pv,YZY → Lu
Y·X,YWX → Qu,YXX → Si,YYX → Pv,YZX → Lu


In [4]:
residues = pd.DataFrame(L.RESIDUES).T
residues["codons"] = [" ".join(L.synonyms(r)) for r in residues.index]
residues

,name,character,real_analogue,stage,codons
Nm,neman,"smallest, flexible",Gly,1,XXY XXX
Kv,korvan,small hydrophobic,Ala,1,XYY XYX
Ds,dresan,acidic,Asp,1,XWY
Th,thulan,branched hydrophobic,Val,1,XZY XZX
My,myrran,"acidic, longer",Glu,2,XWX
Os,osvan,"basic, ring",His,2,YWY
Si,sirran,"basic, long",Arg,2,YXY YXX
Pv,pelvan,"cyclic, rigid",Pro,2,YYY YYX
Qu,quellan,amide,Gln,2,YWX
Lu,lurran,large hydrophobic,Leu,2,YZY YZX


**Reserved (structural) codons.** None of them belongs to the code, so none can appear inside a gene body.
Several are renamed copies of real signals (the promoter is the Pribnow box TATAAT, the RBS is Shine–Dalgarno
AGGAGG). The same two-codon word `WZZ ZZZ` is both the leader's attenuator and the final transcription
terminator.

In [5]:
signals = pd.DataFrame(L.SIGNALS)
signals["codons"] = signals["codons"].map(" ".join)
display(signals)

genes = pd.DataFrame(L.GENES).set_index("name")[
    ["role", "pathway_role", "real_counterpart", "marker", "terminator", "dialect", "register"]]
display(genes)

print(f"{len(L.RESERVED_CODONS)} reserved codons:", " ".join(L.RESERVED_CODONS))
print(f"{len(L.SPACER_ONLY_CODONS)} spacer-only codons:", " ".join(L.SPACER_ONLY_CODONS))

,signal,codons,real_analogue,transcribed
0,promoter,ZWZ WWZ,Pribnow box TATAAT,False
1,rbs,WXX WXX,Shine-Dalgarno AGGAGG,True
2,initiator,WXZ,start codon,True
3,ksensor,WYW YWZ WYW,riboswitch / attenuator sensing domain,True
4,tx_terminator,WZZ ZZZ,rho-independent terminator U-run,True


,role,pathway_role,real_counterpart,marker,terminator,dialect,register
name,,,,,,,
NaxGATE,receptor,brings food Σ in,glucose transporter (GLUT),WWX,ZWW,hydrophobic (membrane protein),X
NaxTAG,intermediate,tags Σ (−1 K),hexokinase,WWY,ZWX,mixed,Y
NaxSNIP,intermediate,cuts into halves (−1 K); valve,PFK + aldolase,WXY,ZXW,"mixed, charged-rich",X
NaxFLIP,intermediate,β → α,triose-phosphate isomerase,WYX,ZYW,"mixed, small",Y
NaxGRAB,intermediate,"oxidation, new phosphate",GAPDH,WYY,ZZW,"mixed, charged-rich",X
NaxPAY,effector,2 K per half,PGK + pyruvate kinase,WZX,ZXZ,mixed,Y
NaxDUMP,effector,empties the bucket → Ω,lactate dehydrogenase,WZY,ZYZ,"mixed, flexible",X


22 reserved codons: WWX WWY WWZ WXX WXY WXZ WYW WYX WYY WZX WZY WZZ YWZ ZWW ZWX ZWZ ZXW ZXZ ZYW ZYZ ZZW ZZZ
26 spacer-only codons: WWW WXW WYZ WZW XWW XWZ XXW XXZ XYW XYZ XZW XZZ YWW YXW YXZ YYW YYZ YZW YZZ ZWY ZXX ZXY ZYX ZYY ZZX ZZY


## 3. Dialects: meaning and style

Each gene writes its body in its own **dialect**, built in two layers:

1. **Meaning — residue composition.** NaxGATE, a membrane transporter, is rich in hydrophobic residues
   (Th, Lu, Kv); NaxSNIP is rich in acidic residues, NaxGRAB in basic ones, NaxFLIP in small ones, NaxDUMP in
   flexible ones (Nm) and amides (Qu).
2. **Style — synonym choice.** For each residue with two codons, a gene prefers one synonym 80/20. There are
   two registers: GATE, SNIP, GRAB and DUMP prefer the synonym ending in X; TAG, FLIP and PAY the one ending in
   Y.

Every weight is positive, so every body codon is used by all seven genes: no single codon gives a gene away.
The table shows the resulting probability of each body codon in each gene.

In [6]:
dialects = pd.DataFrame({g: L.codon_distribution(g) for g in L.GENE_NAMES})
dialects.index = [f"{c} ({L.CODE_TABLE[c]})" for c in dialects.index]
dialects.round(3)

,NaxGATE,NaxTAG,NaxSNIP,NaxFLIP,NaxGRAB,NaxPAY,NaxDUMP
XWY (Ds),0.024,0.231,0.273,0.044,0.044,0.039,0.043
XXY (Nm),0.014,0.046,0.011,0.252,0.012,0.031,0.074
XYY (Kv),0.038,0.185,0.011,0.252,0.012,0.031,0.019
XZY (Th),0.057,0.162,0.008,0.035,0.009,0.031,0.009
XWX (My),0.024,0.040,0.273,0.044,0.044,0.084,0.043
XXX (Nm),0.057,0.012,0.044,0.063,0.050,0.008,0.296
XYX (Kv),0.151,0.046,0.044,0.063,0.050,0.008,0.074
XZX (Th),0.226,0.040,0.031,0.009,0.035,0.008,0.035
YWY (Os),0.024,0.040,0.038,0.044,0.312,0.084,0.043
YXY (Si),0.007,0.046,0.011,0.035,0.050,0.045,0.009


## 4. Spacers

Spacers are noncoding stretches. Their codons are drawn uniformly from all 64 triplets, with two exceptions:

- never the initiator `WXZ` — false starts are placed deliberately (§6 below);
- never a codon that would complete a multi-codon signal (promoter, RBS, K-sensor, terminator pair).

Single reserved codons *may* appear in a spacer: a marker or terminator out of place is the kind of polysemy a
model has to learn to handle.

Every codon becomes a **record** carrying its labels.

In [7]:
SPACER_CODONS = [c for c in L.ALL_CODONS if c != L.INITIATOR]


def record(codon, region, gene=None, start_role=None, residue=None, pos_in_gene=None):
    return {"codon": codon, "region": region, "gene": gene, "start_role": start_role,
            "residue": residue, "pos_in_gene": pos_in_gene}


def completes_motif(codons):
    # True if the last codons of `codons` spell a whole multi-codon signal.
    return any(tuple(codons[-len(m):]) == m for m in L.MULTI_CODON_MOTIFS.values())


def make_spacer(length, region):
    codons = []
    while len(codons) < length:
        candidate = SPACER_CODONS[rng.integers(len(SPACER_CODONS))]
        if not completes_motif(codons + [candidate]):
            codons.append(candidate)
    return [record(c, region) for c in codons]


def draw_length(bounds):
    return int(rng.integers(bounds[0], bounds[1] + 1))


[r["codon"] for r in make_spacer(12, "spacer")]

['WXX',
 'ZWX',
 'YYY',
 'XZW',
 'XZW',
 'ZXZ',
 'WXX',
 'YZW',
 'WZX',
 'WXX',
 'YWY',
 'ZZY']

## 5. Gene blocks

A gene block is

```
RBS (WXX WXX)  gap (1–3)  WXZ  MARKER  BODY (12–24 codons)  TERMINATOR   intergenic (0–2)
```

The body draws its codons from the gene's dialect. With probability 0.15 the RBS is **broken**: one base of
one RBS codon is mutated. The gene is still transcribed, but a ribosome cannot start there, so its `WXZ` is a
*false start* that differs from a true start only in the RBS 2–5 positions upstream. Only mutations that do not
produce another reserved codon are allowed (`WXX` → `XXX`, `YXX`, `ZXX` or `WXW`), so the rest of the gene stays
intact.

In [8]:
RBS_MUTATIONS = [(i, mutant) for i in (0, 1) for mutant in ("XXX", "YXX", "ZXX", "WXW")]
assert not {m for _, m in RBS_MUTATIONS} & set(L.RESERVED_CODONS)


def make_gene_block(gene):
    name = gene["name"]
    rbs = list(L.RBS)
    broken = rng.random() < P_BROKEN_RBS
    if broken:
        i, mutant = RBS_MUTATIONS[rng.integers(len(RBS_MUTATIONS))]
        rbs[i] = mutant

    distribution = L.codon_distribution(gene)
    body = rng.choice(L.CODING_CODONS, size=draw_length(BODY_LENGTH), p=list(distribution.values()))

    core = [record(c, "rbs", name) for c in rbs]
    core += [{**r, "gene": name} for r in make_spacer(draw_length(GAP_LENGTH), "gap")]
    core.append(record(L.INITIATOR, "initiator", name,
                       start_role="false_start_broken_rbs" if broken else "true_start"))
    core.append(record(gene["marker"], "marker", name))
    core += [record(c, "body", name, residue=L.CODE_TABLE[c], pos_in_gene=i) for i, c in enumerate(body)]
    core.append(record(gene["terminator"], "terminator", name))

    return {"gene": name, "core": core,
            "intergenic": make_spacer(draw_length(INTERGENIC_LENGTH), "intergenic")}


block = make_gene_block(L.GENE_BY_NAME["NaxGATE"])
" ".join(r["codon"] for r in block["core"])

'WXX WXX YZZ YZW ZYZ WXZ WWX XZY XZX XZX YZX YZY YZX XZX XYX XZX XYY YZX YZY YYX XYX YZX YZX YZX XXX XZX XYY XWX ZWW'

## 6. Genomes: parts, decoys and assembly

A genome is first generated as **parts**: flanking spacers, promoter, 5'UTR, the seven gene blocks, the
kelate leader (K-sensor + attenuator + a short spacer) and the transcription terminator.

**Spacer decoys.** A Poisson(1) number of bare `WXZ` codons is inserted into the spacers, each into one of
11 slots chosen uniformly (two flanks, the 5'UTR, seven intergenic spacers, the leader spacer), and half of
them are followed by a random gene marker. These are false starts: `WXZ` with no RBS in front.

**Assembly.** The parts are then laid out in an order. The pathway order is GATE, TAG, *leader*, SNIP, FLIP,
GRAB, PAY, DUMP; in the shuffled control the genes are permuted and the leader stays directly before NaxSNIP,
the gene it controls. Because both orders use the same parts, the two corpora differ **only** in gene order.

Signals may sit only where the grammar puts them. Two parts placed side by side can accidentally spell one
(for example an intergenic `WXX` followed by the next gene's `WXX`), so every assembly is checked and, if a
multi-codon signal appears anywhere else, the whole genome is drawn again.

In [9]:
def make_genome_parts():
    parts = {
        "upstream": make_spacer(draw_length(FLANK_LENGTH), "spacer"),
        "promoter": [record(c, "promoter") for c in L.PROMOTER],
        "utr5": make_spacer(draw_length(UTR5_LENGTH), "utr5"),
        "blocks": {g["name"]: make_gene_block(g) for g in L.GENES},
        "leader": {
            "core": [record(c, "ksensor") for c in L.K_SENSOR]
                    + [record(c, "attenuator") for c in L.ATTENUATOR],
            "spacer": make_spacer(draw_length(LEADER_SPACER_LENGTH), "spacer"),
        },
        "tx_terminator": [record(c, "tx_terminator") for c in L.TX_TERMINATOR],
        "downstream": make_spacer(draw_length(FLANK_LENGTH), "spacer"),
    }

    # The 11 spacer slots that can receive a decoy, with the region label their codons carry.
    slots = ([(parts["upstream"], "spacer"), (parts["utr5"], "utr5")]
             + [(parts["blocks"][g]["intergenic"], "intergenic") for g in L.GENE_NAMES]
             + [(parts["leader"]["spacer"], "spacer"), (parts["downstream"], "spacer")])
    for _ in range(rng.poisson(P_SPACER_DECOY)):
        slot, region = slots[rng.integers(len(slots))]
        decoy = [record(L.INITIATOR, region, start_role="false_start_spacer")]
        if rng.random() < P_DECOY_MARKER:
            decoy.append(record(L.GENES[rng.integers(len(L.GENES))]["marker"], region))
        at = int(rng.integers(len(slot) + 1))
        slot[at:at] = decoy
    return parts


def assemble(parts, order):
    genome = parts["upstream"] + parts["promoter"] + parts["utr5"]
    for name in order:
        if name == "NaxSNIP":
            genome += parts["leader"]["core"] + parts["leader"]["spacer"]
        genome += parts["blocks"][name]["core"] + parts["blocks"][name]["intergenic"]
    genome += parts["tx_terminator"] + parts["downstream"]
    return [dict(r, position=i) for i, r in enumerate(genome)]

The check: find every occurrence of every multi-codon signal and compare it with where the grammar intended
one. Intact RBS pairs are intended; so are the promoter, the K-sensor, the attenuator and the transcription
terminator (which spell the same word).

In [10]:
INTENDED_START = {"promoter": {"promoter"}, "rbs": {"rbs"}, "ksensor": {"ksensor"},
                  "tx_terminator": {"attenuator", "tx_terminator"}}


def find_unintended_motifs(genome):
    codons = [r["codon"] for r in genome]
    problems = []
    for name, motif in L.MULTI_CODON_MOTIFS.items():
        k = len(motif)
        for i in range(len(codons) - k + 1):
            if tuple(codons[i:i + k]) != motif:
                continue
            regions = {r["region"] for r in genome[i:i + k]}
            if len(regions) != 1 or not regions <= INTENDED_START[name]:
                problems.append((name, i))
    return problems


def make_genome():
    # Returns (pathway-order genome, shuffled-order genome, number of rejected draws).
    rejected = 0
    while True:
        parts = make_genome_parts()
        shuffled_order = [L.GENE_NAMES[i] for i in rng.permutation(len(L.GENE_NAMES))]
        pathway = assemble(parts, L.GENE_NAMES)
        shuffled = assemble(parts, shuffled_order)
        if not find_unintended_motifs(pathway) and not find_unintended_motifs(shuffled):
            return pathway, shuffled, rejected
        rejected += 1

### Generate the corpus

In [11]:
genomes, genomes_shuffled, rejections = [], [], 0
for genome_id in range(N_GENOMES):
    pathway, shuffled, rejected = make_genome()
    genomes.append(pathway)
    genomes_shuffled.append(shuffled)
    rejections += rejected

lengths = [len(g) for g in genomes]
print(f"{N_GENOMES} genomes; length {min(lengths)}–{max(lengths)} codons (mean {np.mean(lengths):.0f})")
print(f"{rejections} draws rejected because two parts spelled an unintended signal")

400 genomes; length 193–287 codons (mean 245)
85 draws rejected because two parts spelled an unintended signal


## 7. Transcription

Transcription starts at the TSS, right after the promoter (the promoter itself is not transcribed), and copies
the genome with Z replaced by V. Where it stops depends on the condition:

- `K_low`: through the whole operon, to the end of the transcription terminator;
- `K_high`: at the end of the leader's attenuator.

The leader codons are identical in both; only what follows them differs. They are labelled with
`leader_state` (`read_through` or `attenuated`) — the ground truth for case 3.

In [12]:
def transcribe_genome(genome, condition):
    tss = max(r["position"] for r in genome if r["region"] == "promoter") + 1
    stop_region = "tx_terminator" if condition == "K_low" else "attenuator"
    end = max(r["position"] for r in genome if r["region"] == stop_region)
    state = "read_through" if condition == "K_low" else "attenuated"

    transcript = []
    for i, r in enumerate(genome[tss:end + 1]):
        t = dict(r, codon=L.transcribe(r["codon"]), position=i)
        t["leader_state"] = state if r["region"] in ("ksensor", "attenuator") else None
        transcript.append(t)
    return transcript


transcripts = []    # (transcript_id, genome_id, condition, records)
for genome_id, genome in enumerate(genomes):
    for j in range(TRANSCRIPTS_PER_GENOME):
        condition = L.CONDITIONS[rng.integers(len(L.CONDITIONS))]
        transcripts.append((genome_id * TRANSCRIPTS_PER_GENOME + j, genome_id, condition,
                            transcribe_genome(genome, condition)))

conditions = pd.Series([c for _, _, c, _ in transcripts]).value_counts()
print(conditions.to_string())

K_high    844
K_low     756


## 8. Translation

A gene is translated only if its RBS is intact. Its protein is the sequence of residues coded by its body (the
initiator and marker are punctuation, outside the code). Whether a translatable gene is actually *transcribed*
depends on the condition: under `K_high` only NaxGATE and NaxTAG are.

In [13]:
def translate_genes(genome_id, genome):
    proteins = []
    for name in L.GENE_NAMES:
        gene_records = [r for r in genome if r["gene"] == name]
        initiator = next(r for r in gene_records if r["region"] == "initiator")
        translated = initiator["start_role"] == "true_start"
        body = [r["codon"] for r in gene_records if r["region"] == "body"]
        proteins.append({"genome_id": genome_id, "gene": name, "translated": translated,
                         "residues": " ".join(L.translate(body)) if translated else ""})
    return proteins


proteins = pd.DataFrame([p for gid, g in enumerate(genomes) for p in translate_genes(gid, g)])
print(f"{proteins['translated'].mean():.1%} of gene blocks translated "
      f"(expected {1 - P_BROKEN_RBS:.0%}: the rest have a broken RBS)")

84.5% of gene blocks translated (expected 85%: the rest have a broken RBS)


## 9. A look at the corpus

### A genome, coloured by region

In [14]:
REGION_COLOURS = {
    "spacer": "#e6e6e6", "promoter": "#f4a259", "utr5": "#d9d2e9", "rbs": "#8cb369",
    "gap": "#cfe8b8", "initiator": "#bc4b51", "marker": "#5b8e7d", "body": "#f4e285",
    "terminator": "#3d5a80", "intergenic": "#f2f2f2", "ksensor": "#c77dff", "attenuator": "#7b2cbf",
    "tx_terminator": "#0b132b",
}
DARK = {"initiator", "marker", "terminator", "attenuator", "tx_terminator", "ksensor"}


def show_sequence(records, title=""):
    spans = []
    for r in records:
        tip = html.escape(f"{r['position']}: {r['region']} {r['gene'] or ''} {r['start_role'] or ''} "
                          f"{r['residue'] or ''} {r.get('leader_state') or ''}")
        colour = "white" if r["region"] in DARK else "black"
        spans.append(f"<span title='{tip}' style='background:{REGION_COLOURS[r['region']]};color:{colour};"
                     f"font-family:monospace;padding:1px 3px;margin:1px;display:inline-block'>{r['codon']}</span>")
    legend = " ".join(f"<span style='background:{c};color:{'white' if k in DARK else 'black'};"
                      f"padding:1px 4px;font-size:85%'>{k}</span>" for k, c in REGION_COLOURS.items())
    display(HTML(f"<b>{html.escape(title)}</b><div style='line-height:1.9'>{''.join(spans)}</div>"
                 f"<div style='margin-top:4px'>{legend}</div>"))


show_sequence(genomes[0], f"Genome 0 (DNA, pathway order), {len(genomes[0])} codons")

### The same genome transcribed under both conditions

The leader (purple) is the same in both; under `K_high` it is the end of the transcript.

In [15]:
for condition in L.CONDITIONS:
    t = transcribe_genome(genomes[0], condition)
    show_sequence(t, f"Genome 0, {condition} transcript (RNA), {len(t)} codons")

### Proteins of genome 0

In [16]:
for _, p in proteins[proteins["genome_id"] == 0].iterrows():
    if p["translated"]:
        names = "-".join(L.RESIDUES[r]["name"] for r in p["residues"].split())
        print(f"{p['gene']:8} {p['residues']}\n{'':8} {names}\n")
    else:
        print(f"{p['gene']:8} not translated (broken RBS)\n")

NaxGATE  Th Kv Lu My Nm Ds Lu Lu Qu Lu Th Th My Th Lu Th Th
         thulan-korvan-lurran-myrran-neman-dresan-lurran-lurran-quellan-lurran-thulan-thulan-myrran-thulan-lurran-thulan-thulan

NaxTAG   Kv Ds Kv Ds Kv Pv Nm Ds Kv Kv My Th Os My Kv Si Ds Ds Ds
         korvan-dresan-korvan-dresan-korvan-pelvan-neman-dresan-korvan-korvan-myrran-thulan-osvan-myrran-korvan-sirran-dresan-dresan-dresan

NaxSNIP  Os My My Th Kv My Ds Ds Pv Pv Os Ds Pv Qu Os My Ds Ds
         osvan-myrran-myrran-thulan-korvan-myrran-dresan-dresan-pelvan-pelvan-osvan-dresan-pelvan-quellan-osvan-myrran-dresan-dresan

NaxFLIP  Th Nm Pv Th Kv Kv Nm My Nm Nm Kv Ds Kv Kv
         thulan-neman-pelvan-thulan-korvan-korvan-neman-myrran-neman-neman-korvan-dresan-korvan-korvan

NaxGRAB  Si Si Os Si Ds Pv Os Si My Os Os Pv Si Os Os Os Os Os Ds Si Si Nm
         sirran-sirran-osvan-sirran-dresan-pelvan-osvan-sirran-myrran-osvan-osvan-pelvan-sirran-osvan-osvan-osvan-osvan-osvan-dresan-sirran-sirran-neman

NaxPAY   Os Lu Qu Qu Th

### Codon frequencies per gene

The observed body-codon frequencies per gene should approach the dialect probabilities of §3.

In [17]:
body = pd.DataFrame([r for g in genomes for r in g if r["region"] == "body"])
observed = pd.crosstab(body["codon"], body["gene"], normalize="columns")[L.GENE_NAMES]
expected = pd.DataFrame({g: L.codon_distribution(g) for g in L.GENE_NAMES}).loc[observed.index]
print(f"{len(body)} body codons; largest |observed − expected| = {(observed - expected).abs().max().max():.3f}")
observed.round(3)

50422 body codons; largest |observed − expected| = 0.009


gene,NaxGATE,NaxTAG,NaxSNIP,NaxFLIP,NaxGRAB,NaxPAY,NaxDUMP
codon,,,,,,,
XWX,0.024,0.045,0.270,0.043,0.050,0.085,0.045
XWY,0.025,0.235,0.273,0.048,0.041,0.035,0.042
XXX,0.056,0.012,0.044,0.067,0.052,0.009,0.292
XXY,0.015,0.046,0.011,0.243,0.011,0.028,0.077
XYX,0.160,0.048,0.043,0.067,0.050,0.007,0.075
XYY,0.033,0.176,0.010,0.253,0.012,0.031,0.016
XZX,0.228,0.043,0.033,0.009,0.033,0.009,0.031
XZY,0.057,0.161,0.007,0.034,0.008,0.031,0.008
YWX,0.019,0.043,0.039,0.043,0.097,0.199,0.244


## 10. Gene-identifiability check

The dialects must be peaky enough that a gene is identifiable from about 10 body codons, and overlapping
enough that no single codon identifies it (design-spec §2.3). Two measurements:

1. **Single codon.** For each body codon, the posterior P(gene | codon) under a uniform prior, computed from the
   dialect probabilities. Target: never above 0.6.
2. **k codons.** A naive Bayes classifier (add-one smoothing) is trained on the body codons of 80% of the
   genomes and tested on the first k body codons of each gene in the other 20% (split by genome). Target: at
   least 0.90 for k = 10. Chance is 1/7 ≈ 0.14.

In [18]:
P = np.array([[L.codon_distribution(g)[c] for c in L.CODING_CODONS] for g in L.GENE_NAMES])
posterior = P / P.sum(axis=0)
single = pd.DataFrame({"codon": L.CODING_CODONS,
                       "most likely gene": [L.GENE_NAMES[i] for i in posterior.argmax(axis=0)],
                       "P(gene | codon)": posterior.max(axis=0)}).sort_values("P(gene | codon)", ascending=False)
print(f"Highest single-codon posterior: {single['P(gene | codon)'].max():.3f} (target ≤ 0.6)")
single.head(5)

Highest single-codon posterior: 0.595 (target ≤ 0.6)


,codon,most likely gene,P(gene | codon)
13,YXX,NaxGRAB,0.594706
7,XZX,NaxGATE,0.590058
15,YZX,NaxGATE,0.586149
10,YYY,NaxPAY,0.572280
1,XXY,NaxFLIP,0.570550


In [19]:
split_rng = np.random.default_rng(SEED)
genome_ids = split_rng.permutation(N_GENOMES)
train_ids = set(genome_ids[: int(0.8 * N_GENOMES)].tolist())

codon_index = {c: i for i, c in enumerate(L.CODING_CODONS)}
gene_index = {g: i for i, g in enumerate(L.GENE_NAMES)}


def gene_bodies(ids):
    for gid in ids:
        for name in L.GENE_NAMES:
            yield name, [codon_index[r["codon"]] for r in genomes[gid] if r["gene"] == name and r["region"] == "body"]


counts = np.ones((len(L.GENE_NAMES), len(L.CODING_CODONS)))     # add-one smoothing
for name, codons in gene_bodies(sorted(train_ids)):
    np.add.at(counts[gene_index[name]], codons, 1)
log_p = np.log(counts / counts.sum(axis=1, keepdims=True))


def naive_bayes_accuracy(ids, k):
    hits = [log_p[:, codons[:k]].sum(axis=1).argmax() == gene_index[name] for name, codons in gene_bodies(ids)]
    return float(np.mean(hits))


test_ids = sorted(set(range(N_GENOMES)) - train_ids)
identifiability = pd.DataFrame({
    "k body codons": [1, 3, 10],
    "test accuracy": [naive_bayes_accuracy(test_ids, k) for k in (1, 3, 10)],
    "training performance": [naive_bayes_accuracy(sorted(train_ids), k) for k in (1, 3, 10)],
    "chance": 1 / len(L.GENE_NAMES),
})
identifiability

,k body codons,test accuracy,training performance,chance
0,1,0.489286,0.469643,0.142857
1,3,0.687500,0.682589,0.142857
2,10,0.950000,0.950000,0.142857


In [20]:
IDENTIFIABILITY_OK = (single["P(gene | codon)"].max() <= 0.6
                      and identifiability.loc[identifiability["k body codons"] == 10, "test accuracy"].item() >= 0.90)
print("Identifiability targets met:", IDENTIFIABILITY_OK)

Identifiability targets met: True


## 11. Grammar validation

Every rule of the grammar is checked on the whole corpus. Each check prints PASS or FAIL, and the cell fails
if any check fails, so the exported files are known to be correct.

In [21]:
results = []


def check(name, ok):
    results.append((name, bool(ok)))
    print(("PASS  " if ok else "FAIL  ") + name)


all_dna = [r for g in genomes + genomes_shuffled for r in g]
all_rna = [r for _, _, _, t in transcripts for r in t]
body_records = [r for r in all_dna if r["region"] == "body"]

# Reserved-codon rules
check("every body codon is a coding codon and translates to its residue label",
      all(r["codon"] in L.CODE_TABLE and r["residue"] == L.CODE_TABLE[r["codon"]] for r in body_records))
check("no reserved codon occurs in a gene body", not {r["codon"] for r in body_records} & set(L.RESERVED_CODONS))
check("multi-codon signals occur only at their intended places (both orders)",
      all(not find_unintended_motifs(g) for g in genomes + genomes_shuffled))
check("WXZ occurs only as an initiator or as a labelled spacer decoy",
      all(r["region"] == "initiator" or r["start_role"] == "false_start_spacer"
          for r in all_dna if r["codon"] == L.INITIATOR))
check("start_role is set exactly on WXZ codons (WXV in RNA)",
      all((r["start_role"] is not None) == (L.to_dna(r["codon"]) == L.INITIATOR) for r in all_dna + all_rna))


def gene_blocks_ok(genome):
    for name in L.GENE_NAMES:
        g = L.GENE_BY_NAME[name]
        recs = [r for r in genome if r["gene"] == name]
        regions = [r["region"] for r in recs]
        n_gap = regions.count("gap")
        expected = ["rbs", "rbs"] + ["gap"] * n_gap + ["initiator", "marker"] + ["body"] * regions.count("body") + ["terminator"]
        if regions != expected or not (GAP_LENGTH[0] <= n_gap <= GAP_LENGTH[1]):
            return False
        if recs[2 + n_gap + 1]["codon"] != g["marker"] or recs[-1]["codon"] != g["terminator"]:
            return False
        if recs[-1]["position"] - recs[0]["position"] != len(recs) - 1:     # contiguous
            return False
        intact = [r["codon"] for r in recs[:2]] == list(L.RBS)
        if (recs[2 + n_gap]["start_role"] == "true_start") != intact:
            return False
    return True


check("each gene block: RBS, gap 1–3, WXZ, its own marker, body, its own terminator; true start iff RBS intact",
      all(gene_blocks_ok(g) for g in genomes + genomes_shuffled))
check("body lengths are 12–24 and pos_in_gene runs 0..n−1",
      all(BODY_LENGTH[0] <= len(b) <= BODY_LENGTH[1] and [r["pos_in_gene"] for r in b] == list(range(len(b)))
          for g in genomes for name in L.GENE_NAMES
          for b in [[r for r in g if r["gene"] == name and r["region"] == "body"]]))


def gene_order(genome):
    order = []
    for r in genome:
        if r["region"] == "marker" and r["gene"] not in order:
            order.append(r["gene"])
    return order


def leader_before_snip(genome):
    attenuator_end = max(r["position"] for r in genome if r["region"] == "attenuator")
    snip_start = min(r["position"] for r in genome if r["gene"] == "NaxSNIP")
    between = genome[attenuator_end + 1:snip_start]
    return all(r["region"] == "spacer" for r in between)


check("pathway genomes: genes in pathway order, leader directly before NaxSNIP",
      all(gene_order(g) == L.GENE_NAMES and leader_before_snip(g) for g in genomes))
check("shuffled genomes: same parts as their pathway twin, leader directly before NaxSNIP",
      all(leader_before_snip(s) and sorted(r["codon"] for r in s) == sorted(r["codon"] for r in p)
          and all([r["codon"] for r in s if r["gene"] == n] == [r["codon"] for r in p if r["gene"] == n]
                  for n in L.GENE_NAMES)
          for p, s in zip(genomes, genomes_shuffled)))

# Transcription
check("no V in DNA, no Z in RNA",
      all("V" not in r["codon"] for r in all_dna) and all("Z" not in r["codon"] for r in all_rna))


def transcript_ok(genome_id, condition, t):
    g = genomes[genome_id]
    tss = max(r["position"] for r in g if r["region"] == "promoter") + 1
    dna = g[tss:tss + len(t)]
    if [L.transcribe(r["codon"]) for r in dna] != [r["codon"] for r in t]:
        return False
    if [r["region"] for r in dna] != [r["region"] for r in t]:
        return False
    last = "tx_terminator" if condition == "K_low" else "attenuator"
    if [r["region"] for r in t[-2:]] != [last, last]:
        return False
    if last == "attenuator" and any(r["region"] == "attenuator" for r in g[tss + len(t):]):
        return False
    genes_in = {r["gene"] for r in t if r["gene"]}
    return genes_in == (set(L.GENE_NAMES) if condition == "K_low" else {"NaxGATE", "NaxTAG"})


check("each transcript is its genome from the TSS, Z→V, ending at the right terminator for its condition",
      all(transcript_ok(gid, c, t) for _, gid, c, t in transcripts))
check("leader_state is set exactly on leader codons and matches the condition",
      all((r["leader_state"] is not None) == (r["region"] in ("ksensor", "attenuator"))
          and r["leader_state"] in (None, "read_through" if c == "K_low" else "attenuated")
          for _, _, c, t in transcripts for r in t))

# Labels
check("every occurrence has a region from the allowed set",
      all(r["region"] in L.REGIONS for r in all_dna + all_rna))
check("gene and start_role labels are from the allowed sets",
      all(r["gene"] in L.GENE_NAMES + [None] and r["start_role"] in L.START_ROLES + [None]
          for r in all_dna + all_rna))
check("residue and pos_in_gene are present exactly on body codons",
      all((r["residue"] is not None) == (r["region"] == "body") == (r["pos_in_gene"] is not None)
          for r in all_dna + all_rna))

assert all(ok for _, ok in results), "grammar validation failed"
print(f"\nAll {len(results)} checks passed.")

PASS  every body codon is a coding codon and translates to its residue label
PASS  no reserved codon occurs in a gene body


PASS  multi-codon signals occur only at their intended places (both orders)
PASS  WXZ occurs only as an initiator or as a labelled spacer decoy


PASS  start_role is set exactly on WXZ codons (WXV in RNA)
PASS  each gene block: RBS, gap 1–3, WXZ, its own marker, body, its own terminator; true start iff RBS intact
PASS  body lengths are 12–24 and pos_in_gene runs 0..n−1
PASS  pathway genomes: genes in pathway order, leader directly before NaxSNIP


PASS  shuffled genomes: same parts as their pathway twin, leader directly before NaxSNIP
PASS  no V in DNA, no Z in RNA


PASS  each transcript is its genome from the TSS, Z→V, ending at the right terminator for its condition
PASS  leader_state is set exactly on leader codons and matches the condition


PASS  every occurrence has a region from the allowed set


PASS  gene and start_role labels are from the allowed sets
PASS  residue and pos_in_gene are present exactly on body codons

All 15 checks passed.


## 12. Export

**Full corpus (CSV, for the notebooks).** One row per codon occurrence, long format, with the columns of
design-spec §5.2. Absent labels are empty.

In [22]:
GENOME_COLUMNS = ["genome_id", "position", "codon", "region", "gene", "start_role", "residue", "pos_in_gene"]
TRANSCRIPT_COLUMNS = ["transcript_id", "genome_id", "condition", "position", "codon", "region", "gene",
                      "start_role", "leader_state", "residue", "pos_in_gene"]


def genome_frame(genome_list):
    rows = [dict(r, genome_id=gid) for gid, g in enumerate(genome_list) for r in g]
    frame = pd.DataFrame(rows, columns=GENOME_COLUMNS)
    frame["pos_in_gene"] = frame["pos_in_gene"].astype("Int64")
    return frame


genome_df = genome_frame(genomes)
shuffled_df = genome_frame(genomes_shuffled)
transcripts_df = pd.DataFrame(
    [dict(r, transcript_id=tid, genome_id=gid, condition=c) for tid, gid, c, t in transcripts for r in t],
    columns=TRANSCRIPT_COLUMNS)
transcripts_df["pos_in_gene"] = transcripts_df["pos_in_gene"].astype("Int64")

genome_df.to_csv(f"{CORPUS_DIR}/genome.csv", index=False)
shuffled_df.to_csv(f"{CORPUS_DIR}/genome-shuffled.csv", index=False)
transcripts_df.to_csv(f"{CORPUS_DIR}/transcripts.csv", index=False)
proteins.to_csv(f"{CORPUS_DIR}/proteins.csv", index=False)

for name, frame in [("genome.csv", genome_df), ("genome-shuffled.csv", shuffled_df),
                    ("transcripts.csv", transcripts_df), ("proteins.csv", proteins)]:
    print(f"{name:22} {len(frame):>7} rows")
transcripts_df.head()

genome.csv               97991 rows
genome-shuffled.csv      97991 rows
transcripts.csv         194715 rows
proteins.csv              2800 rows


,transcript_id,genome_id,condition,position,codon,region,gene,start_role,leader_state,residue,pos_in_gene
0,0,0,K_high,0,WXV,utr5,NaN,false_start_spacer,NaN,NaN,<NA>
1,0,0,K_high,1,WXX,rbs,NaxGATE,NaN,NaN,NaN,<NA>
2,0,0,K_high,2,WXX,rbs,NaxGATE,NaN,NaN,NaN,<NA>
3,0,0,K_high,3,WXW,gap,NaxGATE,NaN,NaN,NaN,<NA>
4,0,0,K_high,4,VWX,gap,NaxGATE,NaN,NaN,NaN,<NA>


**Web files (JSON, for the labs).** Every JSON file starts with a metadata block (design-spec §5.1), including
the source code of the function that produced it — the labs show it in their Code view. Sequences are stored
column-wise (one array per label) to keep the files small.

- `language.json` — the language, from `protocell_lang.language_as_dict()`.
- `genome-sample.json` — the first 60 genomes, their transcripts (RNA, for the mixed DNA+RNA experiment) and
  the same genomes in shuffled order.
- `showcase-transcripts.json` — six corpus transcripts picked by rule (never edited by hand).

In [23]:
def metadata(function, extra_functions=()):
    return {
        "produced_by": NOTEBOOK,
        "function": function.__name__,
        "source": "\n\n".join(inspect.getsource(f) for f in (function, *extra_functions)),
        "seed": SEED,
        "params": PARAMS,
        "generator_version": L.GENERATOR_VERSION,
        "created": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    }


def columns(records, fields):
    return {f: [r.get(f) for r in records] for f in fields}


def write_json(name, payload):
    path = f"{WEB_DIR}/{name}"
    with open(path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, separators=(",", ":"))
    return os.path.getsize(path)


LABELS = ["codon", "region", "gene", "start_role", "residue", "pos_in_gene"]
sizes = {}

sizes["language.json"] = write_json("language.json", {
    "metadata": metadata(L.language_as_dict), **L.language_as_dict()})

sample_ids = range(N_SAMPLE_GENOMES)
sizes["genome-sample.json"] = write_json("genome-sample.json", {
    "metadata": metadata(make_genome_parts, (make_spacer, make_gene_block, assemble, transcribe_genome)),
    "fields": LABELS,
    "genomes": [{"genome_id": gid, **columns(genomes[gid], LABELS)} for gid in sample_ids],
    "transcripts": [{"transcript_id": tid, "genome_id": gid, "condition": c,
                     **columns(t, LABELS + ["leader_state"])}
                    for tid, gid, c, t in transcripts if gid in sample_ids],
    "shuffled": [{"genome_id": gid, "gene_order": gene_order(genomes_shuffled[gid]),
                  **columns(genomes_shuffled[gid], LABELS)} for gid in sample_ids],
})

The showcase transcripts are chosen by explicit rules, scanning the corpus in order and taking the first
transcript that fits each rule:

In [24]:
def has_broken_rbs(t):
    return any(r["start_role"] == "false_start_broken_rbs" for r in t)


def decoys(t):
    # Positions of spacer decoys, and whether each is followed by a gene marker.
    return [(i, i + 1 < len(t) and L.to_dna(t[i + 1]["codon"]) in L.MARKERS)
            for i, r in enumerate(t) if r["start_role"] == "false_start_spacer"]


def pick_showcase(transcripts):
    by_genome = {}
    for tid, gid, c, t in transcripts:
        by_genome.setdefault(gid, {})[c] = (tid, t)

    def first(rule):
        return next((tid, gid, c, t) for tid, gid, c, t in transcripts if rule(gid, c, t))

    clean = lambda t: not has_broken_rbs(t) and not decoys(t)
    pair_gid = next(gid for gid, d in by_genome.items()
                    if len(d) == 2 and all(clean(t) for _, t in d.values()))
    rules = [
        ("K_low", "Read-through transcript: all seven genes, true starts only", lambda g, c, t: g == pair_gid and c == "K_low"),
        ("K_high", "The same genome attenuated: the leader is the last word", lambda g, c, t: g == pair_gid and c == "K_high"),
        ("broken_rbs", "Read-through with one broken RBS: a WXZ that is not a start",
         lambda g, c, t: c == "K_low" and sum(r["start_role"] == "false_start_broken_rbs" for r in t) == 1 and not decoys(t)),
        ("broken_rbs_K_high", "Attenuated transcript with a broken RBS in NaxGATE or NaxTAG",
         lambda g, c, t: c == "K_high" and has_broken_rbs(t)),
        ("decoy_with_marker", "Spacer decoy: WXZ followed by a gene marker, with no RBS in front",
         lambda g, c, t: c == "K_low" and any(m for _, m in decoys(t)) and not has_broken_rbs(t)),
        ("decoy_bare", "Spacer decoy: a bare WXZ in a spacer",
         lambda g, c, t: c == "K_low" and decoys(t) and not any(m for _, m in decoys(t)) and not has_broken_rbs(t)),
    ]
    showcase = []
    for label, description, rule in rules:
        tid, gid, c, t = first(rule)
        showcase.append({"label": label, "description": description, "transcript_id": tid,
                         "genome_id": gid, "condition": c, **columns(t, LABELS + ["leader_state"])})
    return showcase


showcase = pick_showcase(transcripts)
sizes["showcase-transcripts.json"] = write_json("showcase-transcripts.json", {
    "metadata": metadata(pick_showcase, (transcribe_genome,)),
    "fields": LABELS + ["leader_state"],
    "transcripts": showcase,
})
pd.DataFrame(showcase)[["label", "transcript_id", "genome_id", "condition", "description"]]

,label,transcript_id,genome_id,condition,description
0,K_low,5,1,K_low,"Read-through transcript: all seven genes, true..."
1,K_high,4,1,K_high,The same genome attenuated: the leader is the ...
2,broken_rbs,28,7,K_low,Read-through with one broken RBS: a WXZ that i...
3,broken_rbs_K_high,8,2,K_high,Attenuated transcript with a broken RBS in Nax...
4,decoy_with_marker,25,6,K_low,"Spacer decoy: WXZ followed by a gene marker, w..."
5,decoy_bare,1,0,K_low,Spacer decoy: a bare WXZ in a spacer


## 13. Summary

The web files must respect the size budget of design-spec §5.2: at most 5 MB each and 20 MB in total.

In [25]:
for name, size in sizes.items():
    print(f"{name:28} {size / 1e6:6.2f} MB")
total_web = sum(os.path.getsize(f"{WEB_DIR}/{f}") for f in os.listdir(WEB_DIR) if f.endswith(".json"))
print(f"{'all data/web JSON':28} {total_web / 1e6:6.2f} MB")
assert max(sizes.values()) <= 5e6 and total_web <= 20e6, "size budget exceeded"

for name in ["genome.csv", "genome-shuffled.csv", "transcripts.csv", "proteins.csv"]:
    print(f"{name:28} {os.path.getsize(f'{CORPUS_DIR}/{name}') / 1e6:6.2f} MB")

language.json                  0.01 MB
genome-sample.json             2.29 MB
showcase-transcripts.json      0.04 MB
all data/web JSON              2.34 MB
genome.csv                     2.78 MB
genome-shuffled.csv            2.78 MB
transcripts.csv                8.08 MB
proteins.csv                   0.17 MB
